In [1]:
# 1. Atualizando a biblioteca financeira (garante que vamos puxar os dados mais recentes)
!pip install -q -U yfinance pandas

import yfinance as yf
import pandas as pd

print("🌐 Conectando ao banco de dados financeiro...")

# 2. Definindo a empresa da B3. 
# Regra de Ouro: No yfinance, ações brasileiras precisam do sufixo ".SA"
ticker = "WEGE3.SA"  
empresa = yf.Ticker(ticker)

# 3. Puxando os dados fundamentais
# .balance_sheet traz o Balanço Patrimonial
# .financials traz o DRE (Demonstrativo de Resultados)
balanco_patrimonial = empresa.balance_sheet
dre = empresa.financials

print(f"\n✅ Dados extraídos com sucesso para: {ticker}")

# 4. Limpando a exibição para o nosso Valuation
print("\n📊 Prévia do Balanço Patrimonial (Últimos anos):")
display(balanco_patrimonial.head(10)) # Mostra as 10 primeiras linhas (Ativos, Caixa, etc.)

print("\n📈 Prévia do DRE (Receitas e Lucros):")
display(dre.head(10))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 97.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 re

,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
Treasury Shares Number,1.622025e+06,1.780620e+06,2.083696e+06,1.302592e+06,NaN
Ordinary Shares Number,4.195696e+09,4.195537e+09,4.195234e+09,4.196015e+09,NaN
Share Issued,4.197318e+09,4.197318e+09,4.197318e+09,4.197318e+09,NaN
Total Debt,5.437975e+09,4.418355e+09,3.391960e+09,4.009322e+09,NaN
Tangible Book Value,1.463220e+10,1.938357e+10,1.587084e+10,1.331078e+10,NaN
Invested Capital,2.200801e+10,2.579946e+10,2.017715e+10,1.829449e+10,NaN
Working Capital,9.524444e+09,1.176709e+10,1.034262e+10,9.390333e+09,NaN
Net Tangible Assets,1.463220e+10,1.938357e+10,1.587084e+10,1.331078e+10,NaN
Capital Lease Obligations,8.471530e+08,8.231180e+08,5.568990e+08,5.496300e+08,NaN
Common Stock Equity,1.741718e+10,2.220422e+10,1.734208e+10,1.483480e+10,NaN



📈 Prévia do DRE (Receitas e Lucros):


,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
Tax Effect Of Unusual Items,1.140097e+08,4.842633e+07,7.228352e+07,4.013558e+07,NaN
Tax Rate For Calcs,1.684000e-01,2.010000e-01,1.097000e-01,1.647000e-01,NaN
Normalized EBITDA,8.651314e+09,8.646793e+09,6.709981e+09,5.515876e+09,NaN
Total Unusual Items,6.770170e+08,2.409270e+08,6.589200e+08,2.436890e+08,NaN
Total Unusual Items Excluding Goodwill,6.770170e+08,2.409270e+08,6.589200e+08,2.436890e+08,NaN
Net Income From Continuing Operation Net Minority Interest,6.376219e+09,6.042593e+09,5.731670e+09,4.208084e+09,NaN
Reconciled Depreciation,1.001296e+09,8.124850e+08,6.280420e+08,5.655570e+08,NaN
Reconciled Cost Of Revenue,2.712248e+10,2.517310e+10,2.170274e+10,2.120924e+10,NaN
EBITDA,9.328331e+09,8.887720e+09,7.368901e+09,5.759565e+09,NaN
EBIT,8.327035e+09,8.075235e+09,6.740859e+09,5.194008e+09,NaN


In [2]:
import numpy as np

print("⚙️ Construindo o Motor de Valuation (DCF)...")

# 1. Puxando o Fluxo de Caixa e Dados Complementares
fluxo_caixa = empresa.cashflow
info = empresa.info

# Extraindo métricas chave (usando o último ano disponível como base)
# Tratamento de erro com .get() caso a empresa não tenha a métrica preenchida perfeitamente
caixa_total = info.get('totalCash', 0)
divida_total = info.get('totalDebt', 0)
acoes_em_circulacao = info.get('sharesOutstanding', 1) 

# Simplificação: Pegando o Fluxo de Caixa Livre (Free Cash Flow) mais recente
try:
    fcf_atual = fluxo_caixa.loc['Free Cash Flow'].iloc[0]
except KeyError:
    fcf_atual = fluxo_caixa.loc['Operating Cash Flow'].iloc[0] - fluxo_caixa.loc['Capital Expenditure'].iloc[0]

print(f"💰 Fluxo de Caixa Livre Base: R$ {fcf_atual:,.2f}")
print(f"🏦 Caixa Total: R$ {caixa_total:,.2f} | 📉 Dívida Total: R$ {divida_total:,.2f}")

# 2. Parâmetros do DCF (Premissas Iniciais)
# Aqui é onde o seu LLM vai entrar no futuro para ajustar essas taxas lendo o risco nas notícias!
taxa_crescimento_anos_1_a_5 = 0.08  # Crescimento projetado de 8% ao ano
taxa_crescimento_perpetuidade = 0.03 # Crescimento na perpetuidade (após ano 5) de 3%
wacc = 0.11                         # Taxa de Desconto (Custo Médio Ponderado de Capital) de 11%

# 3. Projetando os Fluxos Futuros (5 anos)
fluxos_projetados = []
fcf_projetado = fcf_atual

for ano in range(1, 6):
    fcf_projetado *= (1 + taxa_crescimento_anos_1_a_5)
    fluxos_projetados.append(fcf_projetado)

# 4. Trazendo a Valor Presente (VP)
vp_fluxos = [f / ((1 + wacc) ** i) for i, f in enumerate(fluxos_projetados, 1)]
soma_vp_fluxos = sum(vp_fluxos)

# 5. Valor Terminal (A perpetuidade trazida a valor presente)
valor_terminal = (fluxos_projetados[-1] * (1 + taxa_crescimento_perpetuidade)) / (wacc - taxa_crescimento_perpetuidade)
vp_valor_terminal = valor_terminal / ((1 + wacc) ** 5)

# 6. Cálculo Final do Preço Justo da Ação
enterprise_value = soma_vp_fluxos + vp_valor_terminal
equity_value = enterprise_value + caixa_total - divida_total
preco_justo = equity_value / acoes_em_circulacao

print("\n" + "="*50)
print(f"🎯 VALUATION CONCLUÍDO: {ticker}")
print("="*50)
print(f"Valor da Empresa (Enterprise Value): R$ {enterprise_value:,.2f}")
print(f"Valor para o Acionista (Equity Value): R$ {equity_value:,.2f}")
print(f"📊 PREÇO JUSTO POR AÇÃO PROJETADO: R$ {preco_justo:.2f}")
print(f"💲 PREÇO ATUAL DE MERCADO: R$ {info.get('currentPrice', 0):.2f}")
print("="*50)

⚙️ Construindo o Motor de Valuation (DCF)...
💰 Fluxo de Caixa Livre Base: R$ 3,759,712,000.00
🏦 Caixa Total: R$ 7,279,864,832.00 | 📉 Dívida Total: R$ 5,437,975,040.00

🎯 VALUATION CONCLUÍDO: WEGE3.SA
Valor da Empresa (Enterprise Value): R$ 59,537,239,314.25
Valor para o Acionista (Equity Value): R$ 61,379,129,106.25
📊 PREÇO JUSTO POR AÇÃO PROJETADO: R$ 14.63
💲 PREÇO ATUAL DE MERCADO: R$ 50.62
